# MLP Probing on Mutopia Embeddings

Probe frozen lilyBERT embeddings (layers 3/6/9/12) for **composer** and **style** classification using a linear MLP with 5-fold stratified cross-validation.

Four embedding configurations are compared:

| Label | Dataset | Description |
|-------|---------|-------------|
| `cb-bm` | `mutopia-cb-bm.json` | CodeBERT finetuned on BaroqueMusic |
| `cb-subset` | `mutopia-cb-subset.json` | CodeBERT on finetuned on a PDMX subset |
| `cb-pretrain` | `mutopia-cb-pretrain_pdmx.json` | CodeBERT after MLM pretraining on PDMX |
| `cb-pdmx-bm` | `mutopia-cb-pdmx-bm.json` | CodeBERT pretrained on PDMX and finetuned on BaroqueMusic |

In [1]:
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, top_k_accuracy_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*ConvergenceWarning.*")

SEED = 42
N_FOLDS = 5
MIN_COUNT = 10
LAYERS = ["layer_3", "layer_6", "layer_9", "layer_12"]
DATA_DIR = Path("../data/mutopia")

DATASETS = {
    "cb-bm": "mutopia-cb-bm.json",
    "cb-pdmx-bm": "mutopia-cb-pdmx-bm.json",
    "cb-pretrain": "mutopia-cb-pretrain_pdmx.json",
    "cb-subset": "mutopia-cb-subset.json",
}

## Helper Functions

In [2]:
def load_dataset(dataset_path):
    """Load dataset JSON and return entries that have embeddings, plus their embedding arrays."""
    with open(dataset_path) as f:
        dataset = json.load(f)
    print(f"Total entries: {len(dataset)}")

    dataset = [entry for entry in dataset if entry.get("embeddings") is not None]
    print(f"Entries with embeddings: {len(dataset)}")

    embeddings = {}
    for layer in LAYERS:
        embs = [np.load(DATA_DIR / entry["embeddings"][layer]) for entry in dataset]
        embeddings[layer] = np.stack(embs)
        print(f"  {layer}: {embeddings[layer].shape}")

    composers = [entry["composer"] for entry in dataset]
    styles = [entry["style"] for entry in dataset]
    return embeddings, composers, styles


def filter_by_min_count(labels, min_count):
    """Return a boolean mask keeping only labels with >= min_count occurrences."""
    counts = Counter(labels)
    valid = {label for label, count in counts.items() if count >= min_count}
    mask = np.array([label in valid for label in labels])
    kept = {l: c for l, c in counts.items() if l in valid}
    dropped = {l: c for l, c in counts.items() if l not in valid}
    print(f"  Kept {len(kept)} classes ({mask.sum()} samples), dropped {len(dropped)} ({(~mask).sum()} samples)")
    return mask


def run_probing(X, y, task_name, layer_name):
    """Train MLP probe with 5-fold stratified CV. Returns metrics dict."""
    n_classes = len(np.unique(y))
    top_ks = [k for k in [1, 3] if k <= n_classes]

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_metrics = {
        "precision": [],
        "recall": [],
        **{f"top{k}": [] for k in top_ks},
    }

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        clf = MLPClassifier(hidden_layer_sizes=(), max_iter=400, random_state=SEED)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_val)
        y_proba = clf.predict_proba(X_val)

        fold_metrics["precision"].append(
            precision_score(y_val, y_pred, average="macro", zero_division=0)
        )
        fold_metrics["recall"].append(
            recall_score(y_val, y_pred, average="macro", zero_division=0)
        )
        for k in top_ks:
            fold_metrics[f"top{k}"].append(
                top_k_accuracy_score(y_val, y_proba, k=k, labels=clf.classes_)
            )

    results = {}
    for metric_name, values in fold_metrics.items():
        results[f"{metric_name}_mean"] = np.mean(values)
        results[f"{metric_name}_std"] = np.std(values)
    return results


def run_all_layers(embeddings, labels, task_name, min_count=MIN_COUNT):
    """Run probing across all layers for a given task. Returns dict of layer -> metrics."""
    mask = filter_by_min_count(labels, min_count)
    filtered_labels = np.array(labels)[mask]

    le = LabelEncoder()
    y = le.fit_transform(filtered_labels)
    print(f"  Classes ({len(le.classes_)}): {list(le.classes_)}")

    layer_results = {}
    for layer in LAYERS:
        layer_results[layer] = run_probing(embeddings[layer][mask], y, task_name, layer)

    return layer_results

## Run Probing Across All Configurations

In [3]:
all_results = {}

for label, filename in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {label} ({filename})")
    print(f"{'='*60}")

    embeddings, composers, styles = load_dataset(DATA_DIR / filename)

    print(f"\n--- Composer Classification ---")
    composer_results = run_all_layers(embeddings, composers, "Composer")

    print(f"\n--- Style Classification ---")
    style_results = run_all_layers(embeddings, styles, "Style")

    all_results[label] = {
        "composer": composer_results,
        "style": style_results,
    }


Dataset: cb-bm (mutopia-cb-bm.json)
Total entries: 2123
Entries with embeddings: 2123
  layer_3: (2123, 768)
  layer_6: (2123, 768)
  layer_9: (2123, 768)
  layer_12: (2123, 768)

--- Composer Classification ---
  Kept 32 classes (1563 samples), dropped 288 (560 samples)
  Classes (32): [np.str_('AguadoD'), np.str_('Anonymous'), np.str_('BachJS'), np.str_('BeethovenLv'), np.str_('BrahmsJ'), np.str_('BurgmullerJFF'), np.str_('CarcassiM'), np.str_('ChopinFF'), np.str_('CzernyC'), np.str_('DiabelliA'), np.str_('FaureG'), np.str_('GiulianiM'), np.str_('GriegE'), np.str_('HandelGF'), np.str_('HaydnFJ'), np.str_('HoretzkyF'), np.str_('JoplinS'), np.str_('KnjzeF'), np.str_('Mendelssohn-BartholdyF'), np.str_('MontePd'), np.str_('MonteverdiC'), np.str_('MozartWA'), np.str_('RachmaninoffS'), np.str_('SatieE'), np.str_('SchubertF'), np.str_('SchumannR'), np.str_('SorF'), np.str_('SousaJP'), np.str_('TitelouzeJ'), np.str_('Traditional'), np.str_('VerdiG'), np.str_('VivaldiA')]


/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimiza


--- Style Classification ---
  Kept 10 classes (2108 samples), dropped 3 (15 samples)
  Classes (10): [np.str_('Baroque'), np.str_('Classical'), np.str_('Folk'), np.str_('Hymn'), np.str_('Jazz'), np.str_('Modern'), np.str_('Renaissance'), np.str_('Romantic'), np.str_('Song'), np.str_('Technique')]


/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimiza


Dataset: cb-pdmx-bm (mutopia-cb-pdmx-bm.json)
Total entries: 2123
Entries with embeddings: 2123
  layer_3: (2123, 768)
  layer_6: (2123, 768)
  layer_9: (2123, 768)
  layer_12: (2123, 768)

--- Composer Classification ---
  Kept 32 classes (1563 samples), dropped 288 (560 samples)
  Classes (32): [np.str_('AguadoD'), np.str_('Anonymous'), np.str_('BachJS'), np.str_('BeethovenLv'), np.str_('BrahmsJ'), np.str_('BurgmullerJFF'), np.str_('CarcassiM'), np.str_('ChopinFF'), np.str_('CzernyC'), np.str_('DiabelliA'), np.str_('FaureG'), np.str_('GiulianiM'), np.str_('GriegE'), np.str_('HandelGF'), np.str_('HaydnFJ'), np.str_('HoretzkyF'), np.str_('JoplinS'), np.str_('KnjzeF'), np.str_('Mendelssohn-BartholdyF'), np.str_('MontePd'), np.str_('MonteverdiC'), np.str_('MozartWA'), np.str_('RachmaninoffS'), np.str_('SatieE'), np.str_('SchubertF'), np.str_('SchumannR'), np.str_('SorF'), np.str_('SousaJP'), np.str_('TitelouzeJ'), np.str_('Traditional'), np.str_('VerdiG'), np.str_('VivaldiA')]


...

/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimiza


Dataset: cb-subset (mutopia-cb-subset.json)
Total entries: 2123
Entries with embeddings: 2123
  layer_3: (2123, 768)
  layer_6: (2123, 768)
  layer_9: (2123, 768)
  layer_12: (2123, 768)

--- Composer Classification ---
  Kept 32 classes (1563 samples), dropped 288 (560 samples)
  Classes (32): [np.str_('AguadoD'), np.str_('Anonymous'), np.str_('BachJS'), np.str_('BeethovenLv'), np.str_('BrahmsJ'), np.str_('BurgmullerJFF'), np.str_('CarcassiM'), np.str_('ChopinFF'), np.str_('CzernyC'), np.str_('DiabelliA'), np.str_('FaureG'), np.str_('GiulianiM'), np.str_('GriegE'), np.str_('HandelGF'), np.str_('HaydnFJ'), np.str_('HoretzkyF'), np.str_('JoplinS'), np.str_('KnjzeF'), np.str_('Mendelssohn-BartholdyF'), np.str_('MontePd'), np.str_('MonteverdiC'), np.str_('MozartWA'), np.str_('RachmaninoffS'), np.str_('SatieE'), np.str_('SchubertF'), np.str_('SchumannR'), np.str_('SorF'), np.str_('SousaJP'), np.str_('TitelouzeJ'), np.str_('Traditional'), np.str_('VerdiG'), np.str_('VivaldiA')]


/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimiza


--- Style Classification ---
  Kept 10 classes (2108 samples), dropped 3 (15 samples)
  Classes (10): [np.str_('Baroque'), np.str_('Classical'), np.str_('Folk'), np.str_('Hymn'), np.str_('Jazz'), np.str_('Modern'), np.str_('Renaissance'), np.str_('Romantic'), np.str_('Song'), np.str_('Technique')]


/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/matteo/projects/csc/MaestroGPT/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimiza

---
## Per-Configuration Summary Tables

In [4]:
def fmt(mean, std):
    return f"{mean:.3f} \u00b1 {std:.3f}"

LAYER_NAMES = {"layer_3": "Layer 3", "layer_6": "Layer 6", "layer_9": "Layer 9", "layer_12": "Layer 12"}

for label, results in all_results.items():
    rows = []
    for layer_key, layer_name in LAYER_NAMES.items():
        c = results["composer"][layer_key]
        s = results["style"][layer_key]
        rows.append({
            "Layer": layer_name,
            "Composer Top-1": fmt(c["top1_mean"], c["top1_std"]),
            "Composer Top-3": fmt(c["top3_mean"], c["top3_std"]),
            "Composer Prec.": fmt(c["precision_mean"], c["precision_std"]),
            "Composer Rec.": fmt(c["recall_mean"], c["recall_std"]),
            "Style Top-1": fmt(s["top1_mean"], s["top1_std"]),
            "Style Top-3": fmt(s["top3_mean"], s["top3_std"]),
            "Style Prec.": fmt(s["precision_mean"], s["precision_std"]),
            "Style Rec.": fmt(s["recall_mean"], s["recall_std"]),
        })

    df = pd.DataFrame(rows).set_index("Layer")
    print(f"\n{'='*40}")
    print(f"  {label}")
    print(f"{'='*40}")
    display(df)


  cb-bm


,Composer Top-1,Composer Top-3,Composer Prec.,Composer Rec.,Style Top-1,Style Top-3,Style Prec.,Style Rec.
Layer,,,,,,,,
Layer 3,0.827 ± 0.012,0.907 ± 0.008,0.753 ± 0.028,0.749 ± 0.018,0.833 ± 0.016,0.959 ± 0.005,0.820 ± 0.057,0.775 ± 0.025
Layer 6,0.829 ± 0.013,0.919 ± 0.013,0.750 ± 0.039,0.752 ± 0.036,0.837 ± 0.004,0.958 ± 0.008,0.813 ± 0.047,0.793 ± 0.016
Layer 9,0.834 ± 0.017,0.919 ± 0.003,0.766 ± 0.043,0.770 ± 0.042,0.832 ± 0.006,0.956 ± 0.008,0.829 ± 0.024,0.794 ± 0.034
Layer 12,0.825 ± 0.015,0.914 ± 0.010,0.756 ± 0.039,0.747 ± 0.039,0.832 ± 0.015,0.958 ± 0.007,0.830 ± 0.025,0.780 ± 0.039



  cb-pdmx-bm


,Composer Top-1,Composer Top-3,Composer Prec.,Composer Rec.,Style Top-1,Style Top-3,Style Prec.,Style Rec.
Layer,,,,,,,,
Layer 3,0.826 ± 0.008,0.912 ± 0.009,0.740 ± 0.013,0.743 ± 0.023,0.830 ± 0.011,0.960 ± 0.008,0.818 ± 0.041,0.786 ± 0.022
Layer 6,0.843 ± 0.010,0.916 ± 0.010,0.765 ± 0.031,0.775 ± 0.028,0.829 ± 0.011,0.948 ± 0.007,0.825 ± 0.013,0.797 ± 0.035
Layer 9,0.839 ± 0.012,0.926 ± 0.010,0.783 ± 0.026,0.771 ± 0.014,0.823 ± 0.009,0.952 ± 0.010,0.806 ± 0.023,0.765 ± 0.027
Layer 12,0.839 ± 0.017,0.916 ± 0.008,0.769 ± 0.034,0.768 ± 0.018,0.821 ± 0.011,0.951 ± 0.009,0.820 ± 0.013,0.776 ± 0.023



  cb-pretrain


,Composer Top-1,Composer Top-3,Composer Prec.,Composer Rec.,Style Top-1,Style Top-3,Style Prec.,Style Rec.
Layer,,,,,,,,
Layer 3,0.807 ± 0.009,0.902 ± 0.005,0.733 ± 0.018,0.737 ± 0.011,0.823 ± 0.010,0.958 ± 0.004,0.813 ± 0.030,0.770 ± 0.027
Layer 6,0.808 ± 0.021,0.904 ± 0.007,0.751 ± 0.025,0.751 ± 0.032,0.826 ± 0.015,0.958 ± 0.005,0.838 ± 0.044,0.781 ± 0.031
Layer 9,0.816 ± 0.016,0.909 ± 0.013,0.736 ± 0.031,0.739 ± 0.024,0.819 ± 0.009,0.947 ± 0.008,0.793 ± 0.040,0.771 ± 0.030
Layer 12,0.803 ± 0.011,0.901 ± 0.011,0.733 ± 0.022,0.731 ± 0.032,0.815 ± 0.015,0.958 ± 0.008,0.829 ± 0.029,0.765 ± 0.027



  cb-subset


,Composer Top-1,Composer Top-3,Composer Prec.,Composer Rec.,Style Top-1,Style Top-3,Style Prec.,Style Rec.
Layer,,,,,,,,
Layer 3,0.811 ± 0.019,0.907 ± 0.005,0.745 ± 0.037,0.738 ± 0.028,0.824 ± 0.010,0.958 ± 0.006,0.820 ± 0.045,0.773 ± 0.030
Layer 6,0.817 ± 0.017,0.907 ± 0.011,0.767 ± 0.035,0.747 ± 0.036,0.823 ± 0.009,0.955 ± 0.009,0.805 ± 0.053,0.775 ± 0.039
Layer 9,0.818 ± 0.018,0.904 ± 0.010,0.761 ± 0.027,0.744 ± 0.026,0.822 ± 0.019,0.954 ± 0.005,0.802 ± 0.034,0.771 ± 0.027
Layer 12,0.816 ± 0.018,0.912 ± 0.010,0.734 ± 0.042,0.735 ± 0.024,0.818 ± 0.012,0.957 ± 0.006,0.805 ± 0.032,0.755 ± 0.038


---
## Cross-Configuration Comparison

Best layer per configuration (by Top-1 accuracy).

In [5]:
rows = []
for label, results in all_results.items():
    for task in ["composer", "style"]:
        best_layer = max(LAYERS, key=lambda l: results[task][l]["top1_mean"])
        r = results[task][best_layer]
        rows.append({
            "Config": label,
            "Task": task.capitalize(),
            "Best Layer": LAYER_NAMES[best_layer],
            "Top-1": fmt(r["top1_mean"], r["top1_std"]),
            "Top-3": fmt(r["top3_mean"], r["top3_std"]),
            "Precision": fmt(r["precision_mean"], r["precision_std"]),
            "Recall": fmt(r["recall_mean"], r["recall_std"]),
        })

comparison_df = pd.DataFrame(rows).set_index(["Config", "Task"])
display(comparison_df)

Best Layer          Top-1          Top-3      Precision  \
Config      Task                                                               
cb-bm       Composer    Layer 9  0.834 ± 0.017  0.919 ± 0.003  0.766 ± 0.043   
            Style       Layer 6  0.837 ± 0.004  0.958 ± 0.008  0.813 ± 0.047   
cb-pdmx-bm  Composer    Layer 6  0.843 ± 0.010  0.916 ± 0.010  0.765 ± 0.031   
            Style       Layer 3  0.830 ± 0.011  0.960 ± 0.008  0.818 ± 0.041   
cb-pretrain Composer    Layer 9  0.816 ± 0.016  0.909 ± 0.013  0.736 ± 0.031   
            Style       Layer 6  0.826 ± 0.015  0.958 ± 0.005  0.838 ± 0.044   
cb-subset   Composer    Layer 9  0.818 ± 0.018  0.904 ± 0.010  0.761 ± 0.027   
            Style       Layer 3  0.824 ± 0.010  0.958 ± 0.006  0.820 ± 0.045   

                             Recall  
Config      Task                     
cb-bm       Composer  0.770 ± 0.042  
            Style     0.793 ± 0.016  
cb-pdmx-bm  Composer  0.775 ± 0.028  
            Style     0.786 ± 0.022  
cb-pretrain Composer  0.739 ± 0.024  
            Style     0.781 ± 0.031  
cb-subset   Composer  0.744 ± 0.026  
            Style     0.773 ± 0.030